## 0. Mettre en place l'environnement

In [1]:
import re
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 120)

print("Environnement prêt.")

Environnement prêt.


## 1. Charger le corpus

In [2]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

texte = urllib.request.urlopen(URL).read().decode("utf-8")

print(f"{len(texte):,} caractères")
print(f"{len(texte.splitlines()):,} lignes")
print("\n--- 300 premiers caractères ---\n")
print(texte[:300])

1,115,394 caractères
40,000 lignes

--- 300 premiers caractères ---

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


## 2. Étiqueter les données

In [3]:
def parser_repliques(txt: str) -> pd.DataFrame:
    """Découpe le corpus en répliques (locuteur, texte)."""
    lignes = []
    for bloc in txt.split("\n\n"):
        bloc = bloc.strip()
        if not bloc:
            continue
        parties = bloc.split("\n")
        entete = re.match(r"^([A-Z][A-Za-z' ]*):$", parties[0])
        if entete and len(parties) > 1:
            lignes.append({
                "locuteur": entete.group(1).strip(),
                "replique": " ".join(p.strip() for p in parties[1:]).strip(),
            })
    return pd.DataFrame(lignes)


df = parser_repliques(texte)

print(f"{len(df):,} répliques extraites")
print(f"{df['locuteur'].nunique()} locuteurs distincts")
df.head(10)

7,094 répliques extraites
297 locuteurs distincts


,locuteur,replique
0,First Citizen,"Before we proceed any further, hear me speak."
1,All,"Speak, speak."
2,First Citizen,You are all resolved rather to die than to fam...
3,All,Resolved. resolved.
4,First Citizen,"First, you know Caius Marcius is chief enemy t..."
5,All,"We know't, we know't."
6,First Citizen,"Let us kill him, and we'll have corn at our ow..."
7,All,"No more talking on't; let it be done: away, away!"
8,Second Citizen,"One word, good citizens."
9,First Citizen,"We are accounted poor citizens, the patricians..."


## 3. Définir la tâche et les choix sur les données

**Objectif** : à partir d'une réplique de *Tiny Shakespeare*, retrouver **quel personnage la prononce** (classification en 5 classes).

### ✅ Choix retenus dans ce corrigé

| Décision | Valeur | Justification |
|---|---|---|
| Nombre de locuteurs | **5** | Effectifs comparables (126 à 196), problème lisible, matrice de confusion 5×5 interprétable à l'œil |
| Seuil de longueur | **≥ 5 mots** | Une réplique du type « Ay, my lord » ne porte aucun signal identifiant : on retire du bruit pur |
| Volume final | **~761 exemples** | Faible, mais suffisant pour un modèle linéaire creux |

In [4]:
N_LOCUTEURS = 5
MIN_MOTS = 5

df["n_mots"] = df["replique"].str.split().str.len()
top_locuteurs = df["locuteur"].value_counts().head(N_LOCUTEURS).index.tolist()

donnees = (
    df[df["locuteur"].isin(top_locuteurs)]
      .query("n_mots >= @MIN_MOTS")
      .loc[:, ["replique", "locuteur"]]
      .reset_index(drop=True)
)

print("Répartition des classes :")
print(donnees["locuteur"].value_counts())

ratio = donnees["locuteur"].value_counts()
print(f"\n{len(donnees)} exemples retenus")
print(f"Ratio classe majoritaire / minoritaire : {ratio.max() / ratio.min():.2f}")
print(f"Part de la classe majoritaire : {ratio.max() / len(donnees):.1%}  <-- plancher de l'accuracy")

Répartition des classes :
locuteur
GLOUCESTER        196
DUKE VINCENTIO    165
ROMEO             141
PETRUCHIO         133
MENENIUS          126
Name: count, dtype: int64

761 exemples retenus
Ratio classe majoritaire / minoritaire : 1.56
Part de la classe majoritaire : 25.8%  <-- plancher de l'accuracy


## 4. Séparer le jeu de données en train / validation / test

In [5]:
# Étape 1 : Séparation du Train (70%) et d'un bloc temporaire (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    donnees["replique"],
    donnees["locuteur"],
    test_size=0.30,                  # On garde 70% pour le train, 30% de côté
    stratify=donnees["locuteur"],    # indispensable : classes déséquilibrées
    random_state=RANDOM_STATE,
)

# Étape 2 : Séparation du bloc temporaire en Validation (15%) et Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,                  # On coupe le bloc de 30% en deux parts égales (15% et 15%)
    stratify=y_temp,                 # indispensable également
    random_state=RANDOM_STATE,
)

print(f"train : {len(X_train)} exemples")
print(f"val   : {len(X_val)} exemples")
print(f"test  : {len(X_test)} exemples")

print("\nProportions par classe (vérification de la stratification) :")
comparaison = pd.DataFrame({
    "train": y_train.value_counts(normalize=True),
    "val": y_val.value_counts(normalize=True),
    "test": y_test.value_counts(normalize=True),
}).round(3)
print(comparaison)

train : 532 exemples
val   : 114 exemples
test  : 115 exemples

Proportions par classe (vérification de la stratification) :
                train    val   test
locuteur                           
GLOUCESTER      0.258  0.254  0.261
DUKE VINCENTIO  0.216  0.219  0.217
ROMEO           0.186  0.184  0.183
PETRUCHIO       0.175  0.175  0.174
MENENIUS        0.165  0.167  0.165


## 5. Embedding

In [6]:
# --- CONFIGURATION & HYPERPARAMÈTRES ---
RANDOM_STATE = 42
T_MAX_LEN = 40  # Notre longueur maximale fine-tunée grâce au graphique

# --- ÉTAPE 1 : DOUBLE SPLIT STRATIFIÉ (70 / 15 / 15) ---
# Séparation Train (70%) et temporaire (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    donnees["replique"],
    donnees["locuteur"],
    test_size=0.30,
    stratify=donnees["locuteur"],  # Indispensable pour classes déséquilibrées
    random_state=RANDOM_STATE,
)

# Séparation du temporaire en Validation (15%) et Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

# --- ÉTAPE 2 : TOKENISATION ET VOCABULAIRE (TRAIN UNIQUEMENT) ---
def tokenizer_simple(texte):
    # Nettoyage léger : mise en minuscules et séparation propre de la ponctuation
    texte = texte.lower()
    texte = re.sub(r"([.,!?])", r" \1 ", texte)
    return texte.split()

# Compter les mots uniquement sur le Train pour éviter tout Data Leakage
compteur_mots = Counter()
for replique in X_train:
    compteur_mots.update(tokenizer_simple(replique))

# Création du dictionnaire d'indexation
# Comme au TP1 (min_df=2), on ne garde que les mots apparaissant au moins 2 fois
vocabulaire = {"[PAD]": 0, "[UNK]": 1}
for mot, freq in compteur_mots.items():
    if freq >= 2:  # Filtre similaire au min_df=2 du TP1
        vocabulaire[mot] = len(vocabulaire)

print(f"Taille du vocabulaire d'entraînement : {len(vocabulaire):,}")

# --- ÉTAPE 3 : CRÉATION DU DATASET PYTORCH ---
class ShakespeareDataset(Dataset):
    def __init__(self, X, y, vocabulaire, label_mapping, max_len=40):
        self.sequences = X.values if hasattr(X, 'values') else X
        self.labels = [label_mapping[lbl] for lbl in y]
        self.vocab = vocabulaire
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        texte = self.sequences[idx]
        label = self.labels[idx]

        # 1. Tokeniser la réplique
        tokens = tokenizer_simple(texte)

        # 2. Convertir en index numériques (avec gestion des mots inconnus [UNK])
        indices = [self.vocab.get(tk, self.vocab["[UNK]"]) for tk in tokens]

        # 3. Appliquer le Padding ou la Troncature jusqu'à T_MAX_LEN
        if len(indices) > self.max_len:
            indices = indices[:self.max_len]  # Troncature
        else:
            indices = indices + [self.vocab["[PAD]"]] * (self.max_len - len(indices))  # Padding

        return torch.tensor(indices, dtype=torch.long), torch.tensor(label, dtype=torch.long)

# Mapping des personnages (classes de sortie) en identifiants uniques (0 à 4)
locuteurs_uniques = sorted(y_train.unique())
label_mapping = {nom: idx for idx, nom in enumerate(locuteurs_uniques)}
print("Mapping des classes :", label_mapping)

# --- ÉTAPE 4 : CRÉATION DES DATALOADERS ---
train_dataset = ShakespeareDataset(X_train, y_train, vocabulaire, label_mapping, max_len=T_MAX_LEN)
val_dataset = ShakespeareDataset(X_val, y_val, vocabulaire, label_mapping, max_len=T_MAX_LEN)
test_dataset = ShakespeareDataset(X_test, y_test, vocabulaire, label_mapping, max_len=T_MAX_LEN)

# Création des DataLoaders par batch de 32 répliques
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Vérification sur un batch
ex_x, ex_y = next(iter(train_loader))
print(f"\nFormat d'un batch d'entrée (X) : {ex_x.shape}")  # Doit être (32, 40)
print(f"Format d'un batch de sortie (y)  : {ex_y.shape}")  # Doit être (32,)

Taille du vocabulaire d'entraînement : 1,427
Mapping des classes : {'DUKE VINCENTIO': 0, 'GLOUCESTER': 1, 'MENENIUS': 2, 'PETRUCHIO': 3, 'ROMEO': 4}

Format d'un batch d'entrée (X) : torch.Size([32, 40])
Format d'un batch de sortie (y)  : torch.Size([32])
